In [11]:
import pandas as pd

df = pd.read_csv("../data/processed/fraud_processed.csv")
df.head()

,user_id,purchase_value,source,browser,sex,age,ip_address,class,hour,lower_bound_ip_address,upper_bound_ip_address,country,time_since_signup,hour_of_day,day_of_week,user_transaction_count,device_transaction_count,ip_transaction_count
0,247547,47,SEO,Safari,F,30,16778864,0,3,16778240.0,16779263.0,Australia,1008.948611,3,6,1,1,1
1,220737,15,SEO,Chrome,F,34,16842045,0,20,16809984.0,16842751.0,Thailand,342.121389,20,2,1,1,1
2,390400,44,Ads,IE,M,29,16843656,0,23,16843264.0,16843775.0,China,554.870556,23,5,1,2,1
3,69592,55,Direct,Chrome,F,30,16938732,0,16,16924672.0,16941055.0,China,2122.471389,16,5,1,1,1
4,174987,51,SEO,Chrome,F,37,16971984,0,4,16941056.0,16973823.0,Thailand,2847.105278,4,1,1,1,1


In [12]:
X = df.drop("class", axis=1)
y = df["class"]

In [13]:
X = pd.get_dummies(X, drop_first=True)

X = pd.get_dummies(X, drop_first=True)

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

SCALE NUMERICAL FEATURES

In [15]:
from sklearn.preprocessing import StandardScaler

num_cols = [
    "purchase_value",
    "age",
    "time_since_signup",
    "user_transaction_count",
    "device_transaction_count",
    "ip_transaction_count"
]

scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

SMOTE

In [16]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

LOGISTIC REGRESSION

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, average_precision_score

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_smote, y_train_smote)

y_pred_lr = lr_model.predict(X_test)
y_proba_lr = lr_model.predict_proba(X_test)[:, 1]

Evaluation

In [18]:
print("Logistic Regression Results")
print("F1 Score:", f1_score(y_test, y_pred_lr))
print("AUC-PR:", average_precision_score(y_test, y_proba_lr))
print(confusion_matrix(y_test, y_pred_lr))

Logistic Regression Results
F1 Score: 0.16487985212569317
AUC-PR: 0.09543579657648502
[[10938 12438]
 [ 1116  1338]]


RANDOM FOREST

In [19]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_smote, y_train_smote)

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

In [20]:
print("Random Forest Results")
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("AUC-PR:", average_precision_score(y_test, y_proba_rf))
print(confusion_matrix(y_test, y_pred_rf))

Random Forest Results
F1 Score: 0.6504520017219113
AUC-PR: 0.7131220902352422
[[22695   681]
 [  943  1511]]


XGBOOST

In [21]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train_smote, y_train_smote)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

In [22]:
print("XGBoost Results")
print("F1 Score:", f1_score(y_test, y_pred_xgb))
print("AUC-PR:", average_precision_score(y_test, y_proba_xgb))
print(confusion_matrix(y_test, y_pred_xgb))

XGBoost Results
F1 Score: 0.6931732933233309
AUC-PR: 0.7011878195187234
[[23217   159]
 [ 1068  1386]]


In [23]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "F1 Score": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb) if 'xgb_model' in globals() else None
    ],
    "AUC-PR": [
        average_precision_score(y_test, y_proba_lr),
        average_precision_score(y_test, y_proba_rf),
        average_precision_score(y_test, y_proba_xgb) if 'xgb_model' in globals() else None
    ]
})

results

,Model,F1 Score,AUC-PR
0,Logistic Regression,0.164880,0.095436
1,Random Forest,0.650452,0.713122
2,XGBoost,0.693173,0.701188


Logistic Regression is used as a baseline model due to its simplicity and interpretability. Random Forest and XGBoost capture non-linear patterns and interactions between behavioral and transactional features. Performance is evaluated using F1-score and AUC-PR due to severe class imbalance.

In [24]:
rf_model.feature_importances_

array([6.25628303e-03, 6.00986640e-03, 5.59316032e-03, 3.96359377e-03,
       6.89092200e-03, 3.61792629e-03, 3.46837381e-03, 1.87037422e-01,
       5.55054008e-03, 7.90042047e-03, 0.00000000e+00, 3.99108247e-01,
       2.74002238e-01, 2.00972703e-02, 1.45154578e-02, 9.60692551e-03,
       9.36803009e-03, 1.35601321e-03, 8.75070845e-03, 1.25162179e-02,
       2.59938289e-06, 1.47193560e-04, 9.34241030e-06, 1.34264468e-08,
       1.04279673e-04, 3.97541219e-05, 2.34495748e-04, 6.60016115e-05,
       2.58234213e-05, 0.00000000e+00, 1.01896000e-07, 1.75004217e-05,
       0.00000000e+00, 1.03425606e-05, 2.64818761e-04, 0.00000000e+00,
       3.26770118e-10, 0.00000000e+00, 0.00000000e+00, 4.74588880e-05,
       1.22503909e-07, 2.72236115e-06, 8.48096992e-07, 1.45150273e-04,
       0.00000000e+00, 0.00000000e+00, 1.67498209e-05, 0.00000000e+00,
       0.00000000e+00, 9.60422802e-09, 2.72660349e-08, 7.85081685e-04,
       0.00000000e+00, 7.09829208e-10, 2.65804592e-04, 6.77138524e-04,
      